# 🍈 Tải & Giải Nén Dataset Sầu Riêng Trực Tiếp Lên Google Drive (Google Colab)

Notebook này chạy trên **Google Colab** để tận dụng hạ tầng mạng tốc độ cao (100 - 300 MB/s của Google):
1. **Tự động Mount Google Drive** của bạn.
2. **Tải lần lượt từng phần dataset từ Zenodo Record: 18603796**.
3. **Tự động giải nén** trực tiếp vào Google Drive.
4. **Tự động xóa file `.zip`** ngay sau khi giải nén xong từng phần để không tốn dung lượng lưu trữ.

> **Lưu ý:** Bạn chỉ cần mở notebook này trên Google Colab và nhấn **Runtime -> Run all** (hoặc Ctrl + F9).

### Bước 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Đường dẫn thư mục lưu dữ liệu trên Google Drive
# Mặc định lưu vào MyDrive/durian-ripeness-cmkd/data/raw
DRIVE_TARGET_DIR = '/content/drive/MyDrive/durian-ripeness-cmkd/data/raw'
os.makedirs(DRIVE_TARGET_DIR, exist_ok=True)

print(f'Thư mục đích trên Google Drive: {DRIVE_TARGET_DIR}')

Mounted at /content/drive
Thư mục đích trên Google Drive: /content/drive/MyDrive/durian-ripeness-cmkd/data/raw


### Bước 2: Cài đặt công cụ tải tăng tốc (tùy chọn)

In [ ]:
# Cài đặt aria2 để tải đa luồng siêu tốc nếu cần
!apt-get install -y -qq aria2
print('Đã sẵn sàng môi trường tải!')

Selecting previously unselected package libcares2:amd64.
(Reading database ... 122807 files and directories currently installed.)
Preparing to unpack .../libcares2_1.27.0-1.0ubuntu1_amd64.deb ...
Unpacking libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.37.0+debian-1build3_amd64.deb ...
Unpacking libaria2-0:amd64 (1.37.0+debian-1build3) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.37.0+debian-1build3_amd64.deb ...
Unpacking aria2 (1.37.0+debian-1build3) ...
Setting up libcares2:amd64 (1.27.0-1.0ubuntu1) ...
Setting up libaria2-0:amd64 (1.37.0+debian-1build3) ...
Setting up aria2 (1.37.0+debian-1build3) ...
Processing triggers for man-db (2.12.0-4build2) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc_proxy.so.2 is not a symbolic link

### Bước 3: Script tự động Tải -> Giải nén vào Drive -> Xóa file nén

In [ ]:
import os
import sys
import json
import time
import subprocess
import urllib.request

RECORD_ID = '18603796'
ZENODO_API_URL = f'https://zenodo.org/api/records/{RECORD_ID}'
LOCAL_TEMP_DIR = '/content/temp_downloads'
os.makedirs(LOCAL_TEMP_DIR, exist_ok=True)

def format_size(size_bytes):
    if size_bytes < 1024 ** 2:
        return f'{size_bytes / 1024:.2f} KB'
    elif size_bytes < 1024 ** 3:
        return f'{size_bytes / (1024 ** 2):.2f} MB'
    else:
        return f'{size_bytes / (1024 ** 3):.2f} GB'

print(f'[*] Đang kết nối Zenodo API: {ZENODO_API_URL} ...')
req = urllib.request.Request(ZENODO_API_URL, headers={'User-Agent': 'Colab-Downloader/1.0'})
with urllib.request.urlopen(req) as resp:
    metadata = json.loads(resp.read().decode('utf-8'))

files = metadata.get('files', [])
print(f'Tìm thấy {len(files)} files trong Record Zenodo:')
for idx, f in enumerate(files, 1):
    print(f'  {idx}. {f["key"]} ({format_size(f["size"])})')

print('\n' + '=' * 60)
print('BẮT ĐẦU QUÁ TRÌNH TẢI, GIẢI NÉN VÀO DRIVE VÀ DỌN DẸP')
print('=' * 60)

for idx, f in enumerate(files, 1):
    fname = f['key']
    fsize = f['size']
    furl = f['links']['self']

    print(f'\n[{idx}/{len(files)}] Đang xử lý: {fname} ({format_size(fsize)})')

    # Nếu là file nhẹ (.csv, .md), tải thẳng vào Google Drive
    if not fname.endswith('.zip'):
        dest_file = os.path.join(DRIVE_TARGET_DIR, fname)
        print(f'    -> Tải trực tiếp vào Drive: {dest_file}')
        subprocess.run(['aria2c', '-x', '16', '-s', '16', '-k', '1M', '-d', DRIVE_TARGET_DIR, '-o', fname, furl], check=True)
        continue

    # Nếu là file .zip nặng:
    # 1. Tải về ổ đĩa cục bộ Colab (/content/temp_downloads) để đạt tốc độ tối đa
    local_zip_path = os.path.join(LOCAL_TEMP_DIR, fname)
    print(f'    1. Tải siêu tốc về ổ đĩa tạm Colab...')
    subprocess.run([
        'aria2c', '-x', '16', '-s', '16', '-k', '1M',
        '-d', LOCAL_TEMP_DIR, '-o', fname, furl
    ], check=True)

    # 2. Giải nén trực tiếp vào thư mục trên Google Drive
    print(f'    2. Đang giải nén trực tiếp vào Google Drive ({DRIVE_TARGET_DIR})...')
    subprocess.run(['unzip', '-q', '-o', local_zip_path, '-d', DRIVE_TARGET_DIR], check=True)
    print(f'    -> Đã giải nén xong {fname} vào Drive.')

    # 3. Xóa ngay file zip để giải phóng ổ cứng Colab
    if os.path.exists(local_zip_path):
        os.remove(local_zip_path)
        print(f'    3. Đã xóa file nén tạm {fname}.')

print('\n' + '=' * 60)
print('🎉 HOÀN THÀNH TOÀN BỘ DATASET ĐÃ ĐƯỢC LƯU TRÊN GOOGLE DRIVE!')
print('=' * 60)

[*] Đang kết nối Zenodo API: https://zenodo.org/api/records/18603796 ...
Tìm thấy 6 files trong Record Zenodo:
  1. README.md (15.62 KB)
  2. dataset_clean_multispectral.zip (928.48 MB)
  3. dataset_clean_sound.zip (1.51 GB)
  4. dataset_clean_thermal.zip (511.49 MB)
  5. dataset_clean_rgb.zip (18.89 GB)
  6. durian_characteristics_cleaned.csv (44.85 KB)

BẮT ĐẦU QUÁ TRÌNH TẢI, GIẢI NÉN VÀO DRIVE VÀ DỌN DẸP

[1/6] Đang xử lý: README.md (15.62 KB)
    -> Tải trực tiếp vào Drive: /content/drive/MyDrive/durian-ripeness-cmkd/data/raw/README.md

[2/6] Đang xử lý: dataset_clean_multispectral.zip (928.48 MB)
    1. Tải siêu tốc về ổ đĩa tạm Colab...
    2. Đang giải nén trực tiếp vào Google Drive (/content/drive/MyDrive/durian-ripeness-cmkd/data/raw)...
    -> Đã giải nén xong dataset_clean_multispectral.zip vào Drive.
    3. Đã xóa file nén tạm dataset_clean_multispectral.zip.

[3/6] Đang xử lý: dataset_clean_sound.zip (1.51 GB)
    1. Tải siêu tốc về ổ đĩa tạm Colab...
    2. Đang giải nén 

### Bước 4: Kiểm tra danh sách thư mục đã giải nén trên Drive

In [ ]:
!ls -la "/content/drive/MyDrive/durian-ripeness-cmkd/data/raw"

total 89
drwx------ 7 root root  4096 Oct 21 15:48 .
drwx------ 3 root root  4096 Oct 21 15:28 ..
drwx------ 2 root root  4096 Oct 21 15:29 dataset_clean_multispectral
drwx------ 2 root root  4096 Feb  2  2026 dataset_clean_rgb
drwx------ 2 root root  4096 Feb  4  2026 dataset_clean_sound
drwx------ 2 root root  4096 Feb 10  2026 dataset_clean_thermal
-rw------- 1 root root 45923 Oct 21 15:48 durian_characteristics_cleaned.csv
drwx------ 3 root root  4096 Oct 21 15:29 __MACOSX
-rw------- 1 root root 15997 Oct 21 15:29 README.md


### 🍈 Bước 5: Phân chia Dataset thành 3 tập Train / Val / Test Nhất Quán (Cross-Modal Stratified Split)

> ⚠️ **NGUYÊN TẮC QUAN TRỌNG VỀ TÍNH NHẤT QUÁN ĐA PHƯƠNG THỨC (Cross-Modal Data Consistency):**
> 1. **Chia ở cấp độ Quả sầu riêng (Sample-level Split):** Tuyệt đối **không chia ngẫu nhiên từng file ảnh hay audio riêng rẽ**. Nếu ảnh góc TOP của quả #45 ở tập Train mà ảnh góc BOT hoặc dòng số liệu cảm biến của quả #45 lại ở tập Test -> sẽ gây **Data Leakage (rò rỉ dữ liệu)** nghiêm trọng và mất tính nhất quán đa phương thức!
> 2. **Đồng bộ hóa 100% giữa các Modalities:** Toàn bộ ảnh RGB (6 góc), file âm thanh gõ (.m4a, .wav), ảnh nhiệt, ma trận nhiệt, ảnh phổ (.tif) và **dòng đặc tính trong `durian_characteristics_cleaned.csv`** của cùng 1 quả sầu riêng **phải cùng thuộc về 1 tập duy nhất (Train, Val hoặc Test)**.
> 3. **Phân tầng (Stratified Split theo nhãn `Ripeness`):** Đảm bảo tỷ lệ các lớp độ chín (`Unripe`, `Ripe`, `Overripe`) được phân bổ đồng đều, cân bằng giữa 3 tập (Mặc định: 70% Train - 15% Val - 15% Test).

> 📂 **Cấu trúc thư mục kết quả (`data/split_data/`):**
> ```text
> data/split_data/
> ├── sample_split_mapping.csv           # Bảng tra cứu toàn bộ 189 quả thuộc tập nào
> ├── train/
> │   ├── dataset_clean_rgb/              # Ảnh RGB của các quả thuộc tập Train
> │   ├── dataset_clean_sound/            # File âm thanh của các quả thuộc tập Train
> │   ├── dataset_clean_thermal/          # File nhiệt của các quả thuộc tập Train
> │   ├── dataset_clean_multispectral/    # File phổ của các quả thuộc tập Train
> │   └── durian_characteristics_cleaned.csv # Bảng đặc tính cảm biến của riêng tập Train
> ├── val/
> │   ├── dataset_clean_rgb/
> │   ├── dataset_clean_sound/
> │   ├── dataset_clean_thermal/
> │   ├── dataset_clean_multispectral/
> │   └── durian_characteristics_cleaned.csv # Bảng đặc tính cảm biến của riêng tập Val
> └── test/
>     ├── dataset_clean_rgb/
>     ├── dataset_clean_sound/
>     ├── dataset_clean_thermal/
>     ├── dataset_clean_multispectral/
>     └── durian_characteristics_cleaned.csv # Bảng đặc tính cảm biến của riêng tập Test
> ```

In [5]:
# ==============================================================================
# SCRIPT PHÂN CHIA DỮ LIỆU ĐA PHƯƠNG THỨC NHẤT QUÁN (CROSS-MODAL STRATIFIED SPLIT)
# Đảm bảo: Ảnh, Âm thanh, Dữ liệu nhiệt, Quang phổ và Bảng đặc tính CSV
# của cùng 1 quả sầu riêng sẽ nằm cùng một thư mục (Train, Val hoặc Test).
# ==============================================================================

import os
import re
import json
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------------------------
# 1. THIẾT LẬP ĐƯỜNG DẪN & TỶ LỆ PHÂN CHIA
# ------------------------------------------------------------------------------
# Tự động phát hiện môi trường: Google Colab hoặc máy cá nhân (Local)
COLAB_RAW_DIR = '/content/drive/MyDrive/durian-ripeness-cmkd/data/raw'
COLAB_SPLIT_DIR = '/content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/'

LOCAL_RAW_DIR = os.path.abspath('data/raw')
LOCAL_SPLIT_DIR = os.path.abspath('data/split_data')

if os.path.exists(COLAB_RAW_DIR):
    RAW_DATA_DIR = COLAB_RAW_DIR
    SPLIT_TARGET_DIR = COLAB_SPLIT_DIR
    print("[*] Phát hiện môi trường Google Colab.")
else:
    # Tìm kiếm đường dẫn local nếu chạy trên máy
    candidates = [
        LOCAL_RAW_DIR,
        os.path.join(os.path.dirname(os.path.abspath('__file__')), 'data', 'raw'),
        '../../../data/raw',
        './'
    ]
    RAW_DATA_DIR = next((p for p in candidates if os.path.exists(os.path.join(p, 'durian_characteristics_cleaned.csv'))), LOCAL_RAW_DIR)
    SPLIT_TARGET_DIR = os.path.abspath(os.path.join(RAW_DATA_DIR, '..', 'split_data'))
    print("[*] Phát hiện môi trường Local.")

print(f"Thư mục dữ liệu gốc (RAW)  : {RAW_DATA_DIR}")
print(f"Thư mục lưu kết quả (SPLIT_DATA): {SPLIT_TARGET_DIR}")

# Tỷ lệ phân chia: 70% Train, 15% Val, 15% Test (Có thể tùy chỉnh theo yêu cầu)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

CSV_FILENAME = 'durian_characteristics_cleaned.csv'
CSV_PATH = os.path.join(RAW_DATA_DIR, CSV_FILENAME)

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Không tìm thấy file bảng đặc tính tại: {CSV_PATH}. Vui lòng kiểm tra lại bước tải dữ liệu!")

# ------------------------------------------------------------------------------
# 2. ĐỌC BẢNG ĐẶC TÍNH VÀ TIẾN HÀNH STRATIFIED SPLIT THEO QUẢ (SAMPLE LEVEL)
# ------------------------------------------------------------------------------
df_raw = pd.read_csv(CSV_PATH)
print(f"\n[*] Đã đọc bảng đặc tính: {len(df_raw)} quả sầu riêng.")

# Trích xuất mã số nguyên sample_id (1 đến 189) từ cột Replicate (VD: 'R1' -> 1)
df_raw['sample_id'] = df_raw['Replicate'].astype(str).str.extract(r'(\d+)')[0].astype(int)

# Bảng tra cứu Code sầu riêng -> sample_id
code_to_id = dict(zip(df_raw['Code'], df_raw['sample_id']))

# Chia phân tầng (Stratified Split) theo nhãn 'Ripeness'
# Bước a: Tách Train (70%) và Phần tạm (30%)
temp_ratio = VAL_RATIO + TEST_RATIO
train_df, temp_df = train_test_split(
    df_raw,
    test_size=temp_ratio,
    stratify=df_raw['Ripeness'],
    random_state=RANDOM_SEED
)

# Bước b: Chia đều Phần tạm (30%) thành Val (15%) và Test (15%)
val_relative_ratio = VAL_RATIO / temp_ratio
val_df, test_df = train_test_split(
    temp_df,
    test_size=(1.0 - val_relative_ratio),
    stratify=temp_df['Ripeness'],
    random_state=RANDOM_SEED
)

# Tập hợp danh sách ID quả cho từng tập
train_sample_ids = set(train_df['sample_id'].tolist())
val_sample_ids = set(val_df['sample_id'].tolist())
test_sample_ids = set(test_df['sample_id'].tolist())

# Kiểm tra tính loại trừ lẫn nhau (Bảo đảm không có quả nào bị trùng lặp)
assert len(train_sample_ids & val_sample_ids) == 0, "Lỗi: Trùng lặp giữa Train và Val!"
assert len(train_sample_ids & test_sample_ids) == 0, "Lỗi: Trùng lặp giữa Train và Test!"
assert len(val_sample_ids & test_sample_ids) == 0, "Lỗi: Trùng lặp giữa Val và Test!"
assert len(train_sample_ids) + len(val_sample_ids) + len(test_sample_ids) == len(df_raw), "Lỗi: Tổng số quả không khớp!"

print(f"\n{'=' * 65}")
print(" KẾT QUẢ PHÂN BỔ SỐ QUẢ VÀ NHÃN RIPENESS THEO TỪNG TẬP ")
print(f"{'=' * 65}")
split_dfs = {'train': train_df, 'val': val_df, 'test': test_df}
split_ids = {'train': train_sample_ids, 'val': val_sample_ids, 'test': test_sample_ids}

for s_name, s_df in split_dfs.items():
    counts = s_df['Ripeness'].value_counts().to_dict()
    print(f"* Tập {s_name.upper():<5}: {len(s_df):>3} quả | Unripe: {counts.get('Unripe', 0):>2}, Ripe: {counts.get('Ripe', 0):>2}, Overripe: {counts.get('Overripe', 0):>2}")
print(f"{'=' * 65}")

# ------------------------------------------------------------------------------
# 3. HÀM NHẬN DIỆN SAMPLE_ID TỪ TÊN FILE CÁC PHƯƠNG THỨC
# ------------------------------------------------------------------------------
def extract_sample_id_from_filename(filename, code_map):
    """Nhận diện sample_id (1..189) cho bất kỳ định dạng file nào trong bộ dữ liệu."""
    # Dạng 1: durian_001_..., durian_045_... (RGB, Thermal JPG/CSV, Multispectral TIF)
    match_durian = re.search(r'durian_0*(\d+)', filename, re.IGNORECASE)
    if match_durian:
        return int(match_durian.group(1))

    # Dạng 2: 45-R.m4a, 1-U.m4a (Acoustic knock iPhone)
    match_iphone = re.match(r'^(\d+)[-_]', filename)
    if match_iphone:
        return int(match_iphone.group(1))

    # Dạng 3: IM_CA_UN_1.wav, MA_CB_R_45.wav (Acoustic knock Android)
    match_android = re.search(r'_(\d+)\.(wav|m4a|mp3)$', filename, re.IGNORECASE)
    if match_android:
        return int(match_android.group(1))

    # Dạng 4: Tra cứu trực tiếp theo tên mã Code
    base_name = os.path.splitext(filename)[0]
    if base_name in code_map:
        return code_map[base_name]

    return None

# ------------------------------------------------------------------------------
# 4. TẠO CẤU TRÚC THƯ MỤC VÀ SAO CHÉP DỮ LIỆU ĐỒNG BỘ
# ------------------------------------------------------------------------------
MODALITY_FOLDERS = [
    'dataset_clean_rgb',
    'dataset_clean_sound',
    'dataset_clean_thermal',
    'dataset_clean_multispectral'
]

# Tạo thư mục train, val, test và các thư mục modality con
for s_name in ['train', 'val', 'test']:
    s_dir = os.path.join(SPLIT_TARGET_DIR, s_name)
    os.makedirs(s_dir, exist_ok=True)
    for m_folder in MODALITY_FOLDERS:
        os.makedirs(os.path.join(s_dir, m_folder), exist_ok=True)

# A. Lưu 3 file CSV đặc tính vào train/, val/, test/ (LOẠI BỎ cột tạm 'sample_id')
print("\n[*] Đang lưu các file bảng đặc tính tương ứng vào từng folder train, val, test...")
for s_name, s_df in split_dfs.items():
    out_csv = os.path.join(SPLIT_TARGET_DIR, s_name, CSV_FILENAME)
    s_df.drop(columns=['sample_id']).to_csv(out_csv, index=False)
    print(f"    -> Đã lưu: {s_name}/{CSV_FILENAME} ({len(s_df)} dòng)")

# Lưu bảng tra cứu tổng hợp sample_split_mapping.csv để tiện tra cứu sau này
mapping_rows = []
for s_name, s_df in split_dfs.items():
    for _, row in s_df.iterrows():
        mapping_rows.append({
            'sample_id': row['sample_id'],
            'Replicate': row['Replicate'],
            'Code': row['Code'],
            'Ripeness': row['Ripeness'],
            'Maturity_Stage': row.get('Maturity_Stage', ''),
            'Split': s_name
        })
mapping_df = pd.DataFrame(mapping_rows).sort_values('sample_id')
mapping_path = os.path.join(SPLIT_TARGET_DIR, 'sample_split_mapping.csv')
mapping_df.to_csv(mapping_path, index=False)
print(f"    -> Đã lưu bảng tra cứu tổng: sample_split_mapping.csv")

# B. Phân loại và sao chép các file thuộc từng Modality
print("\n[*] Đang phân loại và sao chép các file đa phương thức vào train, val, test...")
copy_stats = {s: {m: 0 for m in MODALITY_FOLDERS} for s in ['train', 'val', 'test']}

for m_folder in MODALITY_FOLDERS:
    src_mod_dir = os.path.join(RAW_DATA_DIR, m_folder)
    if not os.path.exists(src_mod_dir):
        print(f"    [!] Bỏ qua '{m_folder}' (Chưa có trong data/raw)")
        continue

    all_files = [f for f in os.listdir(src_mod_dir) if not f.startswith('.')]
    if not all_files:
        print(f"    [!] Bỏ qua '{m_folder}' (Thư mục rỗng)")
        continue

    # Xử lý riêng metadata.csv trong thư mục modality nếu có
    meta_csv_path = os.path.join(src_mod_dir, 'metadata.csv')
    has_meta_csv = os.path.exists(meta_csv_path)
    if has_meta_csv:
        try:
            meta_df = pd.read_csv(meta_csv_path)
            # Xác định cột định danh trong metadata.csv
            id_col = next((c for c in ['sample_num', 'sample_id', 'sample', 'Code'] if c in meta_df.columns), None)
            if id_col:
                meta_df['norm_id'] = meta_df[id_col].astype(str).str.extract(r'(\d+)')[0].astype(float)
                for s_name, s_ids in split_ids.items():
                    sub_meta = meta_df[meta_df['norm_id'].isin(s_ids)].drop(columns=['norm_id'])
                    sub_meta.to_csv(os.path.join(SPLIT_TARGET_DIR, s_name, m_folder, 'metadata.csv'), index=False)
        except Exception as e:
            print(f"    [!] Không thể chia metadata.csv cho {m_folder}: {e}")

    # Sao chép các file dữ liệu (ảnh, âm thanh, ma trận nhiệt, tif)
    copied_mod = 0
    for fname in all_files:
        if fname == 'metadata.csv':
            continue

        sample_id = extract_sample_id_from_filename(fname, code_to_id)
        if sample_id is None:
            continue

        # Xác định tập đích (train, val, test)
        target_split = None
        if sample_id in train_sample_ids:
            target_split = 'train'
        elif sample_id in val_sample_ids:
            target_split = 'val'
        elif sample_id in test_sample_ids:
            target_split = 'test'

        if target_split:
            src_file = os.path.join(src_mod_dir, fname)
            dst_file = os.path.join(SPLIT_TARGET_DIR, target_split, m_folder, fname)
            shutil.copy2(src_file, dst_file)
            copy_stats[target_split][m_folder] += 1
            copied_mod += 1

    print(f"    -> [{m_folder}]: Đã chia xong {copied_mod} files (Train: {copy_stats['train'][m_folder]}, Val: {copy_stats['val'][m_folder]}, Test: {copy_stats['test'][m_folder]})")

# ------------------------------------------------------------------------------
# 5. KIỂM TRA TÍNH NHẤT QUÁN VÀ BÁO CÁO TỔNG KẾT
# ------------------------------------------------------------------------------
print(f"\n{'=' * 75}")
print(" BÁO CÁO TỔNG KẾT PHÂN CHIA DỮ LIỆU ĐA PHƯƠNG THỨC ")
print(f"{'=' * 75}")
print(f"{'Tập dữ liệu':<12} | {'Số quả':<8} | {'CSV Rows':<10} | {'Ảnh RGB':<10} | {'Âm thanh':<10} | {'Ảnh nhiệt':<10} | {'Quang phổ':<10}")
print("-" * 75)
for s in ['train', 'val', 'test']:
    csv_rows = len(split_dfs[s])
    rgb_count = copy_stats[s]['dataset_clean_rgb']
    snd_count = copy_stats[s]['dataset_clean_sound']
    thm_count = copy_stats[s]['dataset_clean_thermal']
    msp_count = copy_stats[s]['dataset_clean_multispectral']
    print(f"{s.upper():<12} | {len(split_ids[s]):<8} | {csv_rows:<10} | {rgb_count:<10} | {snd_count:<10} | {thm_count:<10} | {msp_count:<10}")
print(f"{'=' * 75}")
print("✅ HOÀN TẤT! Dữ liệu đã được phân chia nhất quán tuyệt đối giữa ảnh, âm thanh, nhiệt và bảng đặc tính CSV!")


[*] Phát hiện môi trường Google Colab.
Thư mục dữ liệu gốc (RAW)  : /content/drive/MyDrive/durian-ripeness-cmkd/data/raw
Thư mục lưu kết quả (SPLIT_DATA): /content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/

[*] Đã đọc bảng đặc tính: 189 quả sầu riêng.

 KẾT QUẢ PHÂN BỔ SỐ QUẢ VÀ NHÃN RIPENESS THEO TỪNG TẬP 
* Tập TRAIN: 132 quả | Unripe: 44, Ripe: 44, Overripe: 44
* Tập VAL  :  28 quả | Unripe:  9, Ripe:  9, Overripe: 10
* Tập TEST :  29 quả | Unripe: 10, Ripe: 10, Overripe:  9

[*] Đang lưu các file bảng đặc tính tương ứng vào từng folder train, val, test...
    -> Đã lưu: train/durian_characteristics_cleaned.csv (132 dòng)
    -> Đã lưu: val/durian_characteristics_cleaned.csv (28 dòng)
    -> Đã lưu: test/durian_characteristics_cleaned.csv (29 dòng)
    -> Đã lưu bảng tra cứu tổng: sample_split_mapping.csv

[*] Đang phân loại và sao chép các file đa phương thức vào train, val, test...
    -> [dataset_clean_rgb]: Đã chia xong 2077 files (Train: 1497, Val: 282, Test: 298)
   

### 🔍 Bước 6: Kiểm tra Cấu trúc Thư mục và Phân bố Nhãn sau khi chia

In [6]:
# Kiểm tra trực quan số lượng file và bảng CSV của từng tập trong data/split_data/
import os
import pandas as pd

target_dir = SPLIT_TARGET_DIR if 'SPLIT_TARGET_DIR' in locals() else '/content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/'
print(f"Thư mục kiểm tra: {target_dir}\n")

for split in ['train', 'val', 'test']:
    split_folder = os.path.join(target_dir, split)
    csv_file = os.path.join(split_folder, 'durian_characteristics_cleaned.csv')
    print(f"📂 TẬP [{split.upper()}]:")
    if os.path.exists(csv_file):
        df_s = pd.read_csv(csv_file)
        print(f"   ├── durian_characteristics_cleaned.csv: {len(df_s)} dòng quả sầu riêng")
        print(f"   │   └── Phân bố Ripeness: {df_s['Ripeness'].value_counts().to_dict()}")
    else:
        print("   ├── durian_characteristics_cleaned.csv: ⚠️ Chưa có")

    for mod in ['dataset_clean_rgb', 'dataset_clean_sound', 'dataset_clean_thermal', 'dataset_clean_multispectral']:
        mod_p = os.path.join(split_folder, mod)
        if os.path.exists(mod_p):
            cnt = len([f for f in os.listdir(mod_p) if not f.startswith('.')])
            print(f"   ├── {mod:<30}: {cnt} files")
        else:
            print(f"   ├── {mod:<30}: ⚠️ Chưa tạo")
    print("-" * 60)


Thư mục kiểm tra: /content/drive/MyDrive/durian-ripeness-cmkd/data/split_data/

📂 TẬP [TRAIN]:
   ├── durian_characteristics_cleaned.csv: 132 dòng quả sầu riêng
   │   └── Phân bố Ripeness: {'Overripe': 44, 'Ripe': 44, 'Unripe': 44}
   ├── dataset_clean_rgb             : 1498 files
   ├── dataset_clean_sound           : 407 files
   ├── dataset_clean_thermal         : 2493 files
   ├── dataset_clean_multispectral   : 1681 files
------------------------------------------------------------
📂 TẬP [VAL]:
   ├── durian_characteristics_cleaned.csv: 28 dòng quả sầu riêng
   │   └── Phân bố Ripeness: {'Overripe': 10, 'Unripe': 9, 'Ripe': 9}
   ├── dataset_clean_rgb             : 283 files
   ├── dataset_clean_sound           : 79 files
   ├── dataset_clean_thermal         : 539 files
   ├── dataset_clean_multispectral   : 324 files
------------------------------------------------------------
📂 TẬP [TEST]:
   ├── durian_characteristics_cleaned.csv: 29 dòng quả sầu riêng
   │   └── Phân bố Ripen